# 06: Building and Testing an AI Model for Spam and Ham Message Classification using Appropriate Dataset and Evaluating its Performance

# Step 1: Import Libraries

In [1]:
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 2: Load Dataset

In [3]:
data = pd.read_csv("spam.csv", encoding='latin-1')

data = data[['v1', 'v2']]
data.columns = ['label', 'message']

print(data.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


# Step 3: Convert Labels (Spam/Ham → 1/0)

In [4]:
data['label'] = data['label'].map({'ham':0, 'spam':1})

# Step 4: Split Dataset

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    data['message'],
    data['label'],
    test_size=0.2,
    random_state=42
)

# Step 5: Tokenization

In [6]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Step 6: Padding

In [9]:
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Step 7: Build Model

In [10]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(5000, 64, input_length=100),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

C:\Users\kambl\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# Step 8: Compile Model

In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Step 9: Train Model

In [12]:
model.fit(X_train_pad, y_train, epochs=5, validation_data=(X_test_pad, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - accuracy: 0.8968 - loss: 0.3080 - val_accuracy: 0.9803 - val_loss: 0.0741
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 78ms/step - accuracy: 0.9900 - loss: 0.0422 - val_accuracy: 0.9812 - val_loss: 0.0688
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9946 - loss: 0.0172 - val_accuracy: 0.9812 - val_loss: 0.0688
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.9986 - loss: 0.0068 - val_accuracy: 0.9830 - val_loss: 0.0692
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.9990 - loss: 0.0030 - val_accuracy: 0.9821 - val_loss: 0.0582


#  Step 10: Evaluate Model

In [13]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9875 - loss: 0.0518
Accuracy: 0.9820627570152283


# Step 11: Test Prediction

In [14]:
msg = ["Congratulations! You won a prize"]

seq = tokenizer.texts_to_sequences(msg)
pad = pad_sequences(seq, maxlen=100)

pred = model.predict(pad)

print("Spam" if pred[0][0] > 0.5 else "Ham")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step
Spam
